# ComfyUI on RunPod -- Quickstart

Welcome! This notebook gets you generating images with ComfyUI workflows running on RunPod Serverless GPUs.

**What you'll need:**
- A `.env` file at the project root with `RUNPOD_API_KEY` and `RUNPOD_ENDPOINT_ID`
- Workflow JSON files in `data/comfy-workflows/` (we ship a few to get you started)
- A hydrated network volume with models (see `docs/comfyui-runpod.md` for setup)

**What you'll learn:**
1. Run a workflow with one function call
2. Patch prompts and settings before submission
3. Display and compare outputs
4. Run batch generations across seeds

In [ ]:
# Setup -- run this cell first
import sys, os
os.chdir(os.path.expanduser('~/Workspace/jupyter'))
sys.path.insert(0, 'notebooks/comfy')

from runpod_runner import RunPodWorkflowRunner, run_workflow

## 1. Generate an image in one line

The `run_workflow` helper loads a workflow, sets a seed, submits it to your RunPod endpoint,
polls until complete, and saves the output. That's it -- one function call.

Try changing the seed to get a different result.

In [ ]:
results = run_workflow(
    'data/comfy-workflows/z_image.json',
    seed=42,
    output_dir='outputs/runpod_quickstart',
)

print(f"Success: {results['success']}")
print(f"Files:   {results['files']}")

## 2. Patch a workflow before submission

Sometimes you want to tweak the prompt, change the resolution, or adjust sampler settings
before sending a job. The runner exposes each step so you can modify the workflow dict directly.

Here we'll swap in a custom prompt and bump the step count.

In [ ]:
runner = RunPodWorkflowRunner()
workflow = runner.load_workflow('data/comfy-workflows/z_image.json')

# The positive prompt lives in a CLIPTextEncode node.
# In the z_image workflow that's node "67".
workflow['67']['inputs']['text'] = (
    'a cyberpunk street market at night, neon signs reflecting in rain puddles, '
    'wide angle, cinematic lighting, highly detailed'
)

# Optionally bump steps for higher quality (default is 30)
workflow['69']['inputs']['steps'] = 40

# Set seed and wrap for the API
workflow = runner.update_seed(workflow, seed=7777)
payload = runner.wrap_workflow(workflow)

In [ ]:
# Submit and poll
job_id = runner.submit_job(payload)
status_data = runner.poll_status(job_id, max_polls=120, poll_interval=5)

if status_data and status_data.get('status') == 'COMPLETED':
    files = runner.save_outputs(status_data, 'outputs/runpod_advanced')
    print(f'Saved {len(files)} file(s): {files}')
else:
    print('Job did not complete successfully.')
    print(status_data)

## 3. Display results

View everything that landed in an output directory.

In [ ]:
from IPython.display import display, Image as IPImage
from pathlib import Path

output_dir = Path('outputs/runpod_quickstart')
pngs = sorted(output_dir.glob('*.png'))

if not pngs:
    print('No images found -- run section 1 first.')
else:
    for img_path in pngs:
        print(img_path.name)
        display(IPImage(filename=str(img_path), width=512))

## 4. Batch generation

Sweep across seeds to explore the latent space. Each seed produces a deterministic output,
so you can always reproduce a result you like.

In [ ]:
seeds = [100, 200, 300]
all_files = []

for seed in seeds:
    result = run_workflow(
        'data/comfy-workflows/z_image.json',
        seed=seed,
        output_dir='outputs/batch',
    )
    all_files.extend(result['files'])
    status = 'ok' if result['success'] else 'FAILED'
    print(f'  seed {seed}: {status}')

print(f'\nGenerated {len(all_files)} images total.')

## Next steps

You now know the core loop: load a workflow, optionally patch it, submit, and collect outputs.

**Things to try:**
- Change the resolution by editing node `68` (`EmptySD3LatentImage`) -- try `1024x576` for landscape
- Lower the CFG (`69` -> `inputs.cfg`) to give the model more creative freedom
- Add your own workflow: export API format from the ComfyUI web UI and drop it in `data/comfy-workflows/`

**Reference docs:**
- `docs/comfyui-runpod.md` -- architecture, model management, volume hydration
- `data/comfy-workflows/manifest.json` -- model inventory and HuggingFace sources
- `notebooks/comfy/runpod_runner.py` -- full runner source (audio output supported too)